# Lab 3 — Track B: RAG Offer Generation Demo

End-to-end demo of NorthStar's retention-offer RAG system: section-aware chunking of the
4-document policy corpus (`northstar-policy-docs/`), Titan v2 embeddings in a local FAISS
index, hybrid query retrieval, and generation via Claude Haiku 4.5 on Bedrock. The pipeline's
`invoke(customer_context, question)` call below is the exact interface
`evaluation.evaluation_harness.RAGEvaluator` uses to score it with RAGAS.

In [1]:
from rag.pipeline import invoke
from evaluation.evaluation_harness import OFFER_TEST_CASES

tc = OFFER_TEST_CASES[0]
customer_context = tc.customer_context
question = tc.question
customer_context

{'churn_probability': 0.81, 'loyalty_tier': 'Gold', 'days_since_last_purchase': 47, 'total_lifetime_value': 3120.4, 'avg_order_value': 186.25, 'category_diversity_score': 0.5, 'online_to_store_ratio': 0.72, 'top_categories': ['Outdoor', 'Footwear']}

In [2]:
result = invoke(customer_context, question)
print(result["answer"])

Subject: Your Gold Free Returns + Early Access to New Outdoor Gear

Body:
We've noticed it's been a while since your last visit, and we'd love to welcome you back. As a Gold member, you currently receive free return shipping on all orders—a benefit that makes trying new outdoor gear and footwear risk-free. This week, take advantage of your 24-hour early access to our upcoming sale (live Thursday at 12 PM ET) to shop new arrivals in both categories before general release. Use your early access window to explore fresh styles, and if anything isn't quite right, your free returns mean you can send it back at no cost. Log in to your account now to see what's coming.

Rationale:
This offer anchors to Gold benefits the customer already owns (free return shipping + early sale access), addresses their primary purchase categories (outdoor and footwear), and creates genuine urgency around early access timing rather than artificial scarcity—all without discounting, which research shows resonates b

In [3]:
for ctx in result["contexts"]:
    print(ctx)
    print("---")

[POL-SHP-007 § 2. Free Shipping Thresholds]
Free standard shipping applies at the following order subtotals, after discounts and before tax:

| Tier | Free standard shipping at |
|---|---|
| Non-member | $99 |
| Bronze | $75 |
| Silver | $50 |
| Gold | Any amount |
| Platinum | Any amount |

Free shipping applies to **standard** shipping only. Expedited and overnight are charged at full rate for all tiers, with no exception. Gold and Platinum members do not receive free expedited shipping, and this is the most frequent source of customer confusion on this policy.

Gift card purchases do not count toward the free shipping threshold.

---
---
[POL-LOY-011 § 2. Tiers]
Tier is determined by **trailing 12-month spend**, evaluated on the first day of each month.

| Tier | Trailing 12-month spend | Members (approx.) |
|---|---|---|
| **Bronze** | Under $500 | 61% |
| **Silver** | $500 – $1,999.99 | 26% |
| **Gold** | $2,000 – $4,999.99 | 10% |
| **Platinum** | $5,000 and above | 3% |

Spend c

## Notes

- **3 planted errors corrected** in `rag/prompts.py`'s `TIER_GUIDELINES` (originally in
  `prompt_templates/offer_generation_prompts.md` Template 3): Gold's range was wrongly
  stated as $2,000–$9,999.99 (actual: $2,000–$4,999.99), Platinum's floor was wrongly
  stated as $10,000+ (actual: $5,000+), and a fabricated "Gold gets free expedited shipping
  over $200" line was removed — no tier receives free expedited shipping, per `POL-SHP-007 §2`.
- **Conditional phrasing everywhere**: every tier benefit is phrased as currently-held
  ("as a Gold member, you currently receive...") rather than asserted as permanent fact,
  because Lab 2's `loyalty_tier` (derived from lifetime value) can disagree with
  `POL-LOY-011`'s trailing-12-month-spend definition.
- The 3-way query strategy comparison and known failure modes are in
  `docs/lab3-evidence/track-b-examples.md` and `docs/lab3-evidence/query-strategy-comparison.csv`.
  Model-choice design rationale is in `docs/lab3-model-design.md`.